In [2]:
import json
import numpy as np
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

# =============================================================================
# [1] 설정 (Configuration)
# =============================================================================
# 사용 중인 모델 이름 (ollama list로 확인한 이름과 동일해야 함)
# 추천: "qwen2.5-coder:3b" 또는 "llama3.2:3b"
MODEL_NAME = "smollm2" 
OLLAMA_URL = "http://localhost:11434" # Ollama 기본 주소

# 타겟 스키마 (표준 컬럼 정의)
TARGET_SCHEMA = [
    "user_id", "user_name", "phone_number", "email_address", 
    "signup_date", "last_login", "is_active", "shipping_address",
    "product_code", "category_id"
]

# =============================================================================
# [2] 모델 초기화 (Client Setup)
# =============================================================================
print(f"🔌 Connecting to Ollama Server ({OLLAMA_URL})...")

# 1. 임베딩 모델 (CPU 로컬 실행 - 검색용)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
target_embeddings = embed_model.encode(TARGET_SCHEMA)

# 2. Ollama 클라이언트 (추론용)
# format="json" 옵션이 핵심입니다. 3B 모델도 JSON을 완벽하게 뱉게 만듭니다.
llm = ChatOllama(
    base_url=OLLAMA_URL,
    model=MODEL_NAME,
    temperature=0,      # 사실 기반 매핑이므로 창의성 0으로 설정
    format="json"       # ✨ JSON 강제 출력 모드
)

# =============================================================================
# [3] 프롬프트 템플릿 (Prompt Engineering)
# =============================================================================
template_str = """
You are a Data Engineer. Task: Map the Source Column to the best Target Column.

[Context]
- Source Column Name: "{source_col}"
- Sample Data Value: "{sample_value}"
- Candidate Columns: {candidates}

[Instructions]
1. Analyze the 'Source Column' and 'Sample Data'.
2. Select ONE best match from 'Candidate Columns'.
3. Output MUST be a valid JSON object.

Example JSON:
{{
    "selected_column": "target_name",
    "reason": "Explain why based on sample data"
}}
"""

mapping_prompt = PromptTemplate(
    template=template_str,
    input_variables=["source_col", "sample_value", "candidates"]
)

# =============================================================================
# [4] LangGraph 로직 (Node Definition)
# =============================================================================
class MappingState(TypedDict):
    source_col: str
    sample_value: str
    candidates: List[str]
    final_mapping: str
    reasoning: str

def retriever_node(state: MappingState):
    """Stage 1: 임베딩으로 후보군 3개 압축"""
    print(f"\n🔍 [1. Retriever] Source: '{state['source_col']}' searching...")
    
    source_vec = embed_model.encode([state["source_col"]])
    similarities = cosine_similarity(source_vec, target_embeddings)[0]
    
    # 상위 3개 인덱스
    top_k_indices = np.argsort(similarities)[-3:][::-1]
    candidates = [TARGET_SCHEMA[i] for i in top_k_indices]
    
    print(f"   👉 Candidates: {candidates}")
    return {"candidates": candidates}

def reasoning_node(state: MappingState):
    """Stage 2: Ollama에게 최종 선택 요청"""
    print(f"🧠 [2. Reasoner] Asking Ollama ({MODEL_NAME})...")
    
    # Chain: Prompt -> LLM
    chain = mapping_prompt | llm
    
    try:
        # Ollama 호출
        response = chain.invoke({
            "source_col": state["source_col"],
            "sample_value": state["sample_value"],
            "candidates": state["candidates"]
        })
        
        # ChatOllama는 .content에 응답 문자열이 들어있음
        # format="json" 덕분에 바로 파싱 가능
        result_data = json.loads(response.content)
        
        final_col = result_data.get("selected_column", "Unknown")
        reason = result_data.get("reason", "No reason provided")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        final_col = "Error"
        reason = str(e)
        
    print(f"   ✅ Selected: {final_col}")
    return {"final_mapping": final_col, "reasoning": reason}

# =============================================================================
# [5] 그래프 조립 및 실행
# =============================================================================
workflow = StateGraph(MappingState)

workflow.add_node("retriever", retriever_node)
workflow.add_node("reasoner", reasoning_node)

workflow.set_entry_point("retriever")
workflow.add_edge("retriever", "reasoner")
workflow.add_edge("reasoner", END)

app = workflow.compile()

if __name__ == "__main__":
    # --- 테스트 케이스 ---
    test_input = {
        "source_col": "prod_cat",       # 약어 사용
        "sample_value": "C001-Elec"     # 전자제품 카테고리 코드처럼 보임
    }
    
    print("🚀 Starting Schema Mapping Agent...")
    result = app.invoke(test_input)
    
    print("\n================ FINAL REPORT ================")
    print(f"Input Column : {test_input['source_col']}")
    print(f"Sample Data  : {test_input['sample_value']}")
    print(f"Mapped Target: {result['final_mapping']}")
    print(f"Reasoning    : {result['reasoning']}")
    print("==============================================")

🔌 Connecting to Ollama Server (http://localhost:11434)...
🚀 Starting Schema Mapping Agent...

🔍 [1. Retriever] Source: 'prod_cat' searching...
   👉 Candidates: ['product_code', 'category_id', 'user_name']
🧠 [2. Reasoner] Asking Ollama (smollm2)...
   ✅ Selected: product_code

================ FINAL REPORT ================
Input Column : prod_cat
Sample Data  : C001-Elec
Mapped Target: product_code
Reasoning    : The 'product_code' column is the most suitable target column because it contains unique values that match the Sample Data value "C001-Elec". This matches with the product code in the sample data, and there are no other columns that have a one-to-one mapping with the sample data.
